In [16]:
import pandas as pd
import numpy as np

In [17]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("s1nghayush/data-dump")

print("Path to dataset files:", path)

df = pd.read_csv(path + '/ganjoo_ipl_mbm_bowl_final.csv')

Path to dataset files: /root/.cache/kagglehub/datasets/s1nghayush/data-dump/versions/1


In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14269 entries, 0 to 14268
Data columns (total 16 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   date           14269 non-null  object 
 1   match_id       14269 non-null  int64  
 2   bowl_team      14269 non-null  object 
 3   bat_team       14269 non-null  object 
 4   bowler_id      14269 non-null  int64  
 5   bowler         14269 non-null  object 
 6   inns           14269 non-null  int64  
 7   overs          14269 non-null  object 
 8   runs           14269 non-null  int64  
 9   wickets        14269 non-null  int64  
 10  maiden         14269 non-null  int64  
 11  eco            14269 non-null  float64
 12  balls          14269 non-null  int64  
 13  lbw_or_bowled  14269 non-null  int64  
 14  dots           14269 non-null  int64  
 15  venue          14269 non-null  object 
dtypes: float64(1), int64(9), object(6)
memory usage: 1.7+ MB


In [19]:
# Replace 'DNB' with NaN and convert 'overs' to float
df['overs'] = df['overs'].replace('DNB', np.nan).astype(float)

# Fill NaN values in 'overs' with 0 (if needed)
df['overs'] = df['overs'].fillna(0)

In [20]:
# Replace 'DNB' with NaN and convert 'overs' to float
df['overs'] = df['overs'].replace('DNB', np.nan).astype(float)

# Fill NaN values in 'overs' with 0 (if needed)
df['overs'].fillna(0, inplace=True)

# Calculate dot ball points (+1 per 3 dot balls)
df['dot_ball_points'] = df['dots'] * 1

# Calculate wicket points (+25 per wicket)
df['wicket_points'] = df['wickets'] * 25

# Calculate bonus for LBW/Bowled (+8 per dismissal)
df['bonus_lbw_bowled'] = df['lbw_or_bowled'] * 8

# Wicket milestone bonuses
df['3_wicket_bonus'] = ((df['wickets'] >= 3) & (df['wickets'] < 4)).astype(int) * 4
df['4_wicket_bonus'] = ((df['wickets'] >= 4) & (df['wickets'] < 5)).astype(int) * 8
df['5_wicket_bonus'] = (df['wickets'] >= 5).astype(int) * 12

# Maiden over bonus (+4 per maiden over)
df['maiden_over_bonus'] = df['maiden'] * 12

# Economy rate points calculation function
def calculate_economy_points(row):
    if row['overs'] < 2:  # Minimum 5 overs condition
        return 0
    eco = row['eco']
    if eco < 5:
        return 6
    elif 5 <= eco < 5.99:
        return 4
    elif 6 <= eco < 7:
        return 2
    elif 10 <= eco < 11:
        return -2
    elif 11.01 <= eco < 12:
        return -4
    elif eco >= 12:
        return -6
    return 0  # Default case

# Apply economy points calculation
df['economy_rate_points'] = df.apply(calculate_economy_points, axis=1)

# Total fantasy points calculation
df['bowling_fp'] = (
    df['dot_ball_points'] +
    df['wicket_points'] +
    df['bonus_lbw_bowled'] +
    df['4_wicket_bonus'] +
    df['5_wicket_bonus'] +
    df['3_wicket_bonus'] +
    df['maiden_over_bonus'] +
    df['economy_rate_points']
)

# Drop unnecessary columns
df.drop(columns=[
    'dot_ball_points', 'wicket_points', 'bonus_lbw_bowled',
    '4_wicket_bonus', '5_wicket_bonus', '3_wicket_bonus', 'maiden_over_bonus', 'economy_rate_points'
], inplace=True)

<ipython-input-20-1c252d975b46>:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['overs'].fillna(0, inplace=True)


In [21]:
df = df[['bowler_id', 'match_id', 'bowling_fp', 'date']]

In [24]:
df.to_csv('ipl_bowl_fp.csv', index=False)